# 第 16 章 標本選択とヘックマン: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/econometrics-programming/16-heckit.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
seed = 0             # 乱数の種
n = 3000             # 標本の大きさ
correlation = 0.7    # 選択の誤差と結果の誤差の相関 ρ
repetitions = 100    # 標本を取り直す回数

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
sys.path[:] = [p for p in sys.path if p != str(_root)]
sys.path.insert(0, str(_root))
_lib = sys.modules.get("lib")
if _lib is not None and (
    Path(getattr(_lib, "__file__", "") or "").resolve()
    != (_root / "lib" / "__init__.py").resolve()
):
    for _name in list(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 選択された標本の OLS と、ヘックマンの 2 段階推定（x の真の傾きは 2）

In [ ]:
import numpy as np

from lib.chapter03.multiple import fit_ols
from lib.chapter16.heckit import (
    draw_selection_sample,
    heckit_two_step,
    selection_bias_experiment,
)

df = draw_selection_sample(n, correlation, np.random.default_rng(seed))
chosen = df[df["selected"] == 1]
print(f"選択された割合 {df['selected'].mean():.2f}")
ols = fit_ols(chosen["y"], chosen[["x"]]).coefficients["x"]
print(f"選択された標本の OLS {ols:.3f}")
print(f"ヘックマン {heckit_two_step(df, ['x'], ['x', 'z']).coefficients['x']:.3f}")

estimates = selection_bias_experiment(
    1000, correlation, repetitions, np.random.default_rng(seed)
)
print(estimates.agg(["mean", "std"]).round(3))

## 図: 選択された標本と直線

In [ ]:
from lib.chapter16.plot import plot_selection

plot_selection(df)

## 図: 推定量の分布

In [ ]:
from lib.chapter10.plot import plot_estimate_comparison

plot_estimate_comparison(estimates, 2.0)

## 試してみよう

- `correlation` を 0.0 にすると、選択された標本の OLS も、2.025 と偏りません。
  選択の誤差と結果の誤差が相関して初めて、選択バイアスが生じます
- `correlation` を 0.9 にすると、OLS の平均は 1.828 から 1.781 へ、さらに偏ります。
  ヘックマンは 1.999 で、真の値（2）のままです
- 繰り返した推定のばらつきは、OLS が 0.040、ヘックマンが 0.056 です。
  偏りを取り除く代わりに、ばらつきは少し大きくなります